In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import polars as pl

from src.thetadata_pipeline.settings import get_settings

settings = get_settings()

In [3]:
strangle_trades = pl.read_parquet(get_settings().strategies_dir / 'SPY_strangle_trades.parquet')

# Capital gain

In [146]:
print(
    (strangle_trades['call_profit'] > 0).sum() / len(strangle_trades), strangle_trades['call_profit'].mean(), strangle_trades['call_profit'].sum(), '\n',
    (strangle_trades['put_profit'] > 0).sum() / len(strangle_trades), strangle_trades['put_profit'].mean(), strangle_trades['put_profit'].sum()
)

0.439419795221843 0.04550341296928327 53.33 
 0.42662116040955633 0.06977815699658702 81.77999999999999


In [147]:
print(
    (strangle_trades['strangle_profit'] > 0).mean(),
    strangle_trades['strangle_profit'].mean(),
    strangle_trades['strangle_profit'].sum() / len(strangle_trades),
    strangle_trades['strangle_profit'].sum(),
    ((strangle_trades['put_profit'] > 0) & (strangle_trades['call_profit'] > 0)).sum() / len(strangle_trades)
)

0.6919795221843004 0.1152815699658703 0.1152815699658703 135.10999999999999 0.05631399317406143


In [166]:
invest: float = 200_000
position_margin: int = 15_100
positions = int(invest / position_margin)

capital = (strangle_trades['strangle_profit'] * 100 * positions)
print(np.quantile(capital, np.arange(0, 1.1, 0.1)))

capital[0] = capital[0] + invest
capital = capital.to_pandas().cumsum()
capital.index = strangle_trades['date'].to_list()
print(invest, capital.iloc[-1])

px.line(capital, y=capital, log_y=True).show()
print(
    capital.groupby(pd.to_datetime(capital.index).year).last() - capital.groupby(pd.to_datetime(capital.index).year).first()
)

[-3042.  -1027.   -494.    -13.    143.    260.    410.8   555.1   728.
   949.   4511. ]
200000 375643.0


2021    27001.0
2022    36699.0
2023    29107.0
2024    23738.0
2025    48282.0
2026    13988.0
Name: strangle_profit, dtype: float64


In [72]:
cur_df = strangle_trades\
    [['date', 'avgIV', 'strangle_profit']].to_pandas()\
    .set_index('date')\
    .rolling(window='180D')\
    .mean()

px.line(cur_df, title='Вся динамика капитала зависит от волатильности. Чем она выше, тем выше профит. Раньше эта стратегия была не доступна из-за спредов и малого количество экпираций в неделе.')

# Analyze

In [74]:
strangle_trades = strangle_trades.to_pandas().set_index('date')

In [80]:
df_roll = pd.DataFrame(index=strangle_trades.index)

df_roll['call_profit_share'] = (
        (strangle_trades['call_profit'] > 0).rolling(window='365D').sum() /
        strangle_trades['call_profit'].rolling(window='365D').count()
)
df_roll['put_profit_share'] = (
        (strangle_trades['put_profit'] > 0).rolling(window='365D').sum() /
        strangle_trades['put_profit'].rolling(window='365D').count()
)
df_roll['strangle_profit_share'] = (
        (strangle_trades['strangle_profit'] > 0).rolling(window='365D').sum() /
        strangle_trades['strangle_profit'].rolling(window='365D').count()
)
df_roll['call_put_profit_share'] = (
        ((strangle_trades['put_profit'] > 0) & (strangle_trades['call_profit'] > 0)).rolling(window='365D').sum() / strangle_trades['strangle_profit'].rolling(window='365D').count()
)

df_roll['call_mean'] = strangle_trades['call_profit'].rolling(window='365D').mean()
df_roll['put_mean'] = strangle_trades['put_profit'].rolling(window='365D').mean()
df_roll['strangle_mean'] = strangle_trades['strangle_profit'].rolling(window='365D').mean()

df_roll['call_sum'] = strangle_trades['call_profit'].rolling(window='365D').sum()
df_roll['put_sum'] = strangle_trades['put_profit'].rolling(window='365D').sum()
df_roll['strangle_sum'] = strangle_trades['strangle_profit'].rolling(window='365D').sum()

df_roll['margin'] = (strangle_trades[['strike', 'strike_put']].mean(axis=1) * 0.2).rolling(window='365D').median()
df_roll['cap_gain'] = df_roll['strangle_sum'] / df_roll['margin']

df_roll['exit_call'] = strangle_trades['ext_time_ms'].rolling(window='365D').median()
df_roll['exit_put'] = strangle_trades['ext_time_ms_put'].rolling(window='365D').median()

df_roll['IV_call'] = strangle_trades['IV_bid'].rolling(window='365D').median()
df_roll['IV_put'] = strangle_trades['IV_bid_put'].rolling(window='365D').median()
df_roll['IV_strangle'] = strangle_trades['avgIV'].rolling(window='365D').median()

In [101]:
px.line(
    df_roll[['call_profit_share', 'put_profit_share', 'strangle_profit_share', 'call_put_profit_share']].iloc[30:],
    title='Share of winning'
)

In [102]:
px.line(df_roll[['call_mean', 'put_mean', 'strangle_mean']].iloc[30:], title='Mean profit')

In [105]:
px.line(df_roll[['call_sum', 'put_sum', 'strangle_sum']].iloc[30:], title='Sum of trades')

In [106]:
px.line(df_roll[['cap_gain']].iloc[30:], title='Return on investing capital')

In [107]:
px.line(df_roll[['IV_call', 'IV_put', 'IV_strangle']].iloc[30:], title='IV')

In [108]:
px.line(df_roll[['exit_call', 'exit_put']].iloc[50:], title='TimeOfExit')

In [100]:
import plotly.graph_objects as go

cur_df = df_roll.iloc[30:]
fig = go.Figure()

fig.add_trace(go.Scatter(y=cur_df['IV_strangle'], x=cur_df.index, name='IV_strangle', yaxis='y1'))
fig.add_trace(go.Scatter(y=cur_df['cap_gain'], x=cur_df.index, name='Return on investing capital', yaxis='y2'))
fig.add_trace(go.Scatter(y=cur_df['strangle_sum'], x=cur_df.index, name='strangle_sum', yaxis='y3'))
fig.add_trace(go.Scatter(y=cur_df['strangle_mean'], x=cur_df.index, name='strangle_mean', yaxis='y4'))
fig.add_trace(go.Scatter(y=cur_df['strangle_profit_share'], x=cur_df.index, name='strangle_profit_share', yaxis='y5'))

fig.update_layout(
    title='Combined Plot with Multiple Y-Axes',
    height=500,
    xaxis=dict(title='Date', domain=[0.12, 0.9], tickfont=dict(size=10)),
    yaxis=dict(title='IV', side='left',  position=0.12, showgrid=False, tickfont=dict(size=10)),
    yaxis2=dict(title='Return on Capital', overlaying='y', side='right', position=0.9, showgrid=False, tickfont=dict(size=10)),
    yaxis3=dict(title='Sum of Trades', overlaying='y', side='left', anchor='free', position=0.00, showgrid=False, tickfont=dict(size=10)),
    yaxis4=dict(title='Mean Profit', overlaying='y', side='right', anchor='free', position=0.95, showgrid=False, tickfont=dict(size=10)),
    yaxis5=dict(title='Share of Winning', overlaying='y', side='left', anchor='free', position=0.07, showgrid=False, tickfont=dict(size=10))
)
fig.show()

# Time

In [112]:
print(
    '\n',np.quantile(strangle_trades['ext_time_ms'].dropna(), np.arange(0, 1.1, 0.1)),
    '\n',np.quantile(strangle_trades['ext_time_ms_put'].dropna(), np.arange(0, 1.1, 0.1))
)


 [34661820.  35293610.6 35681913.8 36120754.8 36778473.4 37577533.
 39014323.8 40886052.  45274539.2 51188193.2 57597953. ] 
 [34715302.  35268750.5 35595911.8 36031881.9 36426517.4 37255949.5
 38302386.2 40293008.1 44678032.2 49996390.6 57594934. ]


In [117]:
cur_df = pd.DataFrame(data=[strangle_trades['ext_time_ms'].dropna().to_list(), strangle_trades['ext_time_ms_put'].dropna().to_list()]).T
cur_df.columns = ['ext_time_ms', 'ext_time_ms_put']
px.histogram(cur_df.melt(), x="value", color="variable", nbins=20, opacity=0.5, width=1000, histnorm='percent', barmode='overlay')

# IV

In [130]:
strangle_trades['ext_time_ms'] = strangle_trades['ext_time_ms'].fillna(0)
strangle_trades['ext_time_ms_put'] = strangle_trades['ext_time_ms_put'].fillna(0)

strangle_trades['spread'] = (strangle_trades['ent_opt_ask'] - strangle_trades['ent_opt_bid']) / strangle_trades['ent_opt_ask']
strangle_trades['spread_put'] = (strangle_trades['ent_opt_ask_put'] - strangle_trades['ent_opt_bid_put']) / strangle_trades['ent_opt_ask_put']
strangle_trades['spread_avg'] = (strangle_trades['spread'] + strangle_trades['spread_put']) / 2

In [124]:
groups = pd.qcut(strangle_trades['IV_bid'], q=np.arange(0, 1.1, 0.1))
cur_box = strangle_trades.groupby(groups).agg(
    mean=('call_profit', np.mean),
    median=('call_profit', np.median),
    sum=('call_profit', sum),
    count=('call_profit', len),
    positive_count=('call_profit', lambda x: (x > 0).sum()),
)
cur_box['bin_profit'] = cur_box['positive_count'] / cur_box['count']
cur_box['day_profit'] = cur_box['sum'] / cur_box['count']
cur_box.style.background_gradient(cmap='RdYlGn', axis=0)

,mean,median,sum,count,positive_count,bin_profit,day_profit
IV_bid,,,,,,,
"(0.07919999999999999, 0.148]",0.049492,0.020000,5.840000,118,60,0.508475,0.049492
"(0.148, 0.17]",0.067521,0.070000,7.900000,117,62,0.529915,0.067521
"(0.17, 0.189]",0.045556,0.000000,5.330000,117,58,0.495726,0.045556
"(0.189, 0.21]",0.049145,-0.080000,5.750000,117,53,0.452991,0.049145
"(0.21, 0.231]",0.069316,-0.120000,8.110000,117,53,0.452991,0.069316
"(0.231, 0.266]",0.065299,-0.200000,7.640000,117,49,0.418803,0.065299
"(0.266, 0.307]",0.096838,-0.200000,11.330000,117,50,0.427350,0.096838
"(0.307, 0.354]",0.032735,-0.170000,3.830000,117,49,0.418803,0.032735
"(0.354, 0.457]",-0.017179,-0.290000,-2.010000,117,39,0.333333,-0.017179


In [125]:
groups = pd.qcut(strangle_trades['IV_bid_put'], q=np.arange(0, 1.1, 0.1))
cur_box = strangle_trades.groupby(groups).agg(
   mean=('put_profit', np.mean),
    median=('put_profit', np.median),
    sum=('put_profit', sum),
    count=('put_profit', len),
    positive_count=('put_profit', lambda x: (x > 0).sum()),
)
cur_box['bin_profit'] = cur_box['positive_count'] / cur_box['count']
cur_box['day_profit'] = cur_box['sum'] / cur_box['count']
cur_box.style.background_gradient(cmap='RdYlGn', axis=0)

,mean,median,sum,count,positive_count,bin_profit,day_profit
IV_bid_put,,,,,,,
"(0.0904, 0.168]",0.038729,-0.025000,4.570000,118,57,0.483051,0.038729
"(0.168, 0.186]",0.017607,-0.170000,2.060000,117,49,0.418803,0.017607
"(0.186, 0.206]",0.000940,-0.200000,0.110000,117,48,0.410256,0.000940
"(0.206, 0.228]",0.058547,-0.100000,6.850000,117,52,0.444444,0.058547
"(0.228, 0.255]",0.048120,-0.190000,5.630000,117,53,0.452991,0.048120
"(0.255, 0.287]",0.083761,-0.190000,9.800000,117,48,0.410256,0.083761
"(0.287, 0.326]",0.078889,-0.280000,9.230000,117,50,0.427350,0.078889
"(0.326, 0.366]",0.041282,-0.340000,4.830000,117,44,0.376068,0.041282
"(0.366, 0.483]",0.159829,-0.140000,18.700000,117,52,0.444444,0.159829


In [126]:
groups = pd.qcut(strangle_trades['avgIV'], q=np.arange(0, 1.1, 0.1))
cur_box = strangle_trades.groupby(groups).agg(
   mean=('strangle_profit', np.mean),
    median=('strangle_profit', np.median),
    sum=('strangle_profit', sum),
    count=('strangle_profit', len),
    positive_count=('strangle_profit', lambda x: (x > 0).sum()),
)
cur_box['bin_profit'] = cur_box['positive_count'] / cur_box['count']
cur_box['day_profit'] = cur_box['sum'] / cur_box['count']
cur_box.style.background_gradient(cmap='RdYlGn', axis=0)

,mean,median,sum,count,positive_count,bin_profit,day_profit
avgIV,,,,,,,
"(0.0848, 0.161]",0.132542,0.150000,15.640000,118,92,0.779661,0.132542
"(0.161, 0.179]",0.096752,0.120000,11.320000,117,80,0.683761,0.096752
"(0.179, 0.199]",0.108120,0.130000,12.650000,117,84,0.717949,0.108120
"(0.199, 0.217]",0.090513,0.150000,10.590000,117,80,0.683761,0.090513
"(0.217, 0.245]",0.115470,0.240000,13.510000,117,78,0.666667,0.115470
"(0.245, 0.276]",0.103590,0.240000,12.120000,117,85,0.726496,0.103590
"(0.276, 0.317]",0.144188,0.290000,16.870000,117,83,0.709402,0.144188
"(0.317, 0.358]",0.109573,0.330000,12.820000,117,79,0.675214,0.109573
"(0.358, 0.47]",0.103419,0.390000,12.100000,117,77,0.658120,0.103419


# Spreads

In [131]:
groups = pd.qcut(strangle_trades['spread'], q=np.arange(0, 1.1, 0.1), duplicates='drop')
cur_box = strangle_trades.groupby(groups).agg(
   mean=('call_profit', np.mean),
    median=('call_profit', np.median),
    sum=('call_profit', sum),
    count=('call_profit', len),
    positive_count=('call_profit', lambda x: (x > 0).sum()),
)
cur_box['bin_profit'] = cur_box['positive_count'] / cur_box['count']
cur_box['day_profit'] = cur_box['sum'] / cur_box['count']
cur_box.style.background_gradient(cmap='RdYlGn', axis=0)

,mean,median,sum,count,positive_count,bin_profit,day_profit
spread,,,,,,,
"(-0.312, 0.00862]",-0.002000,-0.310000,-0.240000,120,44,0.366667,-0.002000
"(0.00862, 0.0103]",0.118305,-0.205000,13.960000,118,47,0.398305,0.118305
"(0.0103, 0.0122]",0.085431,-0.185000,9.910000,116,43,0.370690,0.085431
"(0.0122, 0.0139]",0.059913,-0.180000,6.890000,115,47,0.408696,0.059913
"(0.0139, 0.0156]",0.002000,-0.190000,0.260000,130,49,0.376923,0.002000
"(0.0156, 0.0175]",0.033365,-0.200000,3.470000,104,45,0.432692,0.033365
"(0.0175, 0.0196]",0.004167,-0.165000,0.550000,132,50,0.378788,0.004167
"(0.0196, 0.0219]",0.035294,0.000000,3.600000,102,51,0.500000,0.035294
"(0.0219, 0.0263]",0.075000,0.230000,8.850000,118,68,0.576271,0.075000


In [132]:
groups = pd.qcut(strangle_trades['spread_put'], q=np.arange(0, 1.1, 0.1), duplicates='drop')
cur_box = strangle_trades.groupby(groups).agg(
   mean=('put_profit', np.mean),
    median=('put_profit', np.median),
    sum=('put_profit', sum),
    count=('put_profit', len),
    positive_count=('put_profit', lambda x: (x > 0).sum()),
)
cur_box['bin_profit'] = cur_box['positive_count'] / cur_box['count']
cur_box['day_profit'] = cur_box['sum'] / cur_box['count']
cur_box.style.background_gradient(cmap='RdYlGn', axis=0)

,mean,median,sum,count,positive_count,bin_profit,day_profit
spread_put,,,,,,,
"(-0.0682, 0.0082]",0.233136,-0.275000,27.510000,118,52,0.440678,0.233136
"(0.0082, 0.01]",0.029583,-0.250000,3.550000,120,43,0.358333,0.029583
"(0.01, 0.0114]",0.086311,-0.240000,10.530000,122,49,0.401639,0.086311
"(0.0114, 0.0128]",0.068609,-0.240000,7.890000,115,46,0.400000,0.068609
"(0.0128, 0.0141]",0.106283,-0.220000,12.010000,113,49,0.433628,0.106283
"(0.0141, 0.0156]",-0.019274,-0.210000,-2.390000,124,47,0.379032,-0.019274
"(0.0156, 0.0175]",0.045304,-0.160000,5.210000,115,50,0.434783,0.045304
"(0.0175, 0.02]",0.047563,-0.110000,5.660000,119,54,0.453782,0.047563
"(0.02, 0.0238]",0.004336,-0.160000,0.490000,113,48,0.424779,0.004336


In [133]:
groups = pd.qcut(strangle_trades['spread_avg'], q=np.arange(0, 1.1, 0.1))
cur_box = strangle_trades.groupby(groups).agg(
   mean=('strangle_profit', np.mean),
    median=('strangle_profit', np.median),
    sum=('strangle_profit', sum),
    count=('strangle_profit', len),
    positive_count=('strangle_profit', lambda x: (x > 0).sum()),
)
cur_box['bin_profit'] = cur_box['positive_count'] / cur_box['count']
cur_box['day_profit'] = cur_box['sum'] / cur_box['count']
cur_box.style.background_gradient(cmap='RdYlGn', axis=0)

,mean,median,sum,count,positive_count,bin_profit,day_profit
spread_avg,,,,,,,
"(-0.19, 0.00896]",0.131610,0.395000,15.530000,118,76,0.644068,0.131610
"(0.00896, 0.0108]",0.054017,0.290000,6.320000,117,72,0.615385,0.054017
"(0.0108, 0.0123]",0.194872,0.440000,22.800000,117,82,0.700855,0.194872
"(0.0123, 0.0136]",0.083846,0.240000,9.810000,117,78,0.666667,0.083846
"(0.0136, 0.0151]",0.130256,0.240000,15.240000,117,84,0.717949,0.130256
"(0.0151, 0.0166]",0.091026,0.190000,10.650000,117,82,0.700855,0.091026
"(0.0166, 0.0186]",0.066496,0.180000,7.780000,117,80,0.683761,0.066496
"(0.0186, 0.0205]",0.141538,0.130000,16.560000,117,88,0.752137,0.141538
"(0.0205, 0.0247]",0.140000,0.185000,16.520000,118,92,0.779661,0.140000


In [139]:
strangle_trades['spread_avg_roll'] = strangle_trades['spread_avg'].rolling(50, min_periods=1).mean()

px.line(strangle_trades[['spread_avg', 'spread_avg_roll']])

# Premium to price

In [141]:
strangle_trades['premium_to_baPrice'] = (
        strangle_trades[['ent_opt_bid', 'ent_opt_bid_put']].sum(axis=1) /
        strangle_trades[['strike', 'strike_put']].mean(axis=1)
)

In [142]:
groups = pd.qcut(strangle_trades['premium_to_baPrice'], q=np.arange(0, 1.1, 0.1))
cur_box = strangle_trades.groupby(groups).agg(
   mean=('strangle_profit', np.mean),
    median=('strangle_profit', np.median),
    sum=('strangle_profit', sum),
    count=('strangle_profit', len),
    positive_count=('strangle_profit', lambda x: (x > 0).sum()),
)
cur_box['bin_profit'] = cur_box['positive_count'] / cur_box['count']
cur_box['day_profit'] = cur_box['sum'] / cur_box['count']
cur_box.style.background_gradient(cmap='RdYlGn', axis=0)

,mean,median,sum,count,positive_count,bin_profit,day_profit
premium_to_baPrice,,,,,,,
"(-0.00021500000000000002, 0.00174]",0.092966,0.090000,10.970000,118,80,0.677966,0.092966
"(0.00174, 0.00208]",0.117179,0.130000,13.710000,117,88,0.752137,0.117179
"(0.00208, 0.00234]",0.141880,0.170000,16.600000,117,92,0.786325,0.141880
"(0.00234, 0.00267]",0.056949,0.145000,6.720000,118,77,0.652542,0.056949
"(0.00267, 0.003]",0.122931,0.190000,14.260000,116,82,0.706897,0.122931
"(0.003, 0.00349]",0.097521,0.240000,11.410000,117,81,0.692308,0.097521
"(0.00349, 0.00404]",0.097436,0.320000,11.400000,117,79,0.675214,0.097436
"(0.00404, 0.00458]",0.205556,0.390000,24.050000,117,88,0.752137,0.205556
"(0.00458, 0.006]",0.020769,0.370000,2.430000,117,70,0.598291,0.020769
